# CUPED в A/B-тесте

Сравниваем оценку эффекта по исходной метрике и после поправки CUPED.
Данные синтетические. В каждой группе по 8 000 пользователей, а истинный
эффект, заданный в генераторе, равен 0,25 единицы метрики.
Ковариата `x_pre` измерена до эксперимента.

Запустите ячейки сверху вниз. После каждого задания вызывается автопроверка.

In [ ]:
import numpy as np
import pandas as pd
from math import sqrt

rng = np.random.default_rng(123)

n_per_group = 8_000
n = 2 * n_per_group
variant = np.array(['control'] * n_per_group + ['treatment'] * n_per_group)
is_treatment = (variant == 'treatment').astype(int)

x_pre = rng.lognormal(mean=1.8, sigma=0.75, size=n)

post_activity = 0.8 * x_pre + 0.6 * is_treatment + rng.normal(0, 1.0, size=n)

noise = rng.normal(0, 3.5, size=n)
true_effect = 0.25
y = 3.0 + 0.65 * x_pre + true_effect * is_treatment + noise

exp = pd.DataFrame({
    'user_id': np.arange(1, n + 1),
    'variant': variant,
    'x_pre': x_pre,
    'post_activity': post_activity,
    'y': y,
})

exp.head()

In [ ]:
#@title 🔒 Автопроверка CUPED — запустите один раз и не редактируйте
import numpy as _np
import pandas as _pd
from math import sqrt as _sqrt


def _effect_and_se(data, metric):
    c = data.loc[data['variant'] == 'control', metric]
    t = data.loc[data['variant'] == 'treatment', metric]
    effect = t.mean() - c.mean()
    se = _sqrt(t.var(ddof=1) / len(t) + c.var(ddof=1) / len(c))
    return effect, se


def _theta_reference(data, x_col='x_pre', y_col='y'):
    x = data[x_col]
    y = data[y_col]
    x_centered = x - x.mean()
    y_centered = y - y.mean()
    cov_y_x = (x_centered * y_centered).sum() / (len(data) - 1)
    var_x = (x_centered ** 2).sum() / (len(data) - 1)
    return cov_y_x / var_x


def check_task(task_id, **kwargs):
    if task_id == 1:
        theta = kwargs.get('theta')
        ref = _theta_reference(exp)
        assert isinstance(theta, (int, float, _np.floating)), 'theta должен быть числом.'
        assert abs(theta - ref) < 1e-6, 'theta посчитан неверно. Используйте Cov(Y, X) / Var(X).'
        print('✅ Задача 1 зачтена: theta посчитан корректно.')

    elif task_id == 2:
        y_cuped = kwargs.get('y_cuped')
        assert y_cuped is not None, 'Передайте y_cuped в check_task.'
        y_cuped = _pd.Series(y_cuped)
        assert len(y_cuped) == len(exp), 'y_cuped должен иметь ту же длину, что exp.'
        ref_theta = _theta_reference(exp)
        ref = exp['y'] - ref_theta * (exp['x_pre'] - exp['x_pre'].mean())
        assert _np.allclose(y_cuped.values, ref.values), 'Формула y_cuped неверна. Проверьте центрирование X.'
        print('✅ Задача 2 зачтена: CUPED-метрика построена корректно.')

    elif task_id == 3:
        y_cuped = kwargs.get('y_cuped')
        data = exp.copy()
        data['y_cuped'] = y_cuped
        _, se_base = _effect_and_se(data, 'y')
        _, se_cuped = _effect_and_se(data, 'y_cuped')
        assert se_cuped < se_base, 'SE после CUPED должен быть меньше baseline SE.'
        reduction = 1 - se_cuped / se_base
        assert reduction > 0.15, 'Снижение SE слишком маленькое: проверьте формулу CUPED.'
        print(f'✅ Задача 3 зачтена: SE снизился на {reduction:.1%}.')

    elif task_id == 4:
        answer = str(kwargs.get('answer', '')).lower()
        explanation = str(kwargs.get('explanation', '')).lower()
        text = answer + ' ' + explanation
        assert 'post_activity' in text, 'Нужно явно назвать post_activity.'
        assert any(word in text for word in ['после', 'post', 'treatment', 'эксперимент']), 'Объясните, что признак измерен после начала эксперимента или зависит от treatment.'
        assert any(word in text for word in ['нельзя', 'некоррект', 'смещ', 'bias', 'invalid']), 'Нужно явно написать, что такой covariate использовать нельзя.'
        print('✅ Задача 4 зачтена: риск post-treatment covariate объяснен.')

    elif task_id == 5:
        rho = kwargs.get('rho')
        predicted_reduction = kwargs.get('predicted_reduction')
        ref_rho = exp['x_pre'].corr(exp['y'])
        ref_red = ref_rho ** 2
        assert abs(rho - ref_rho) < 1e-6, 'rho должен быть corr(x_pre, y).'
        assert abs(predicted_reduction - ref_red) < 1e-6, 'Ожидаемое снижение дисперсии равно rho^2.'
        print('✅ Задача 5 зачтена: связь корреляции и снижения дисперсии посчитана.')

    elif task_id == 6:
        report = kwargs.get('report')
        assert isinstance(report, _pd.DataFrame), 'report должен быть pandas DataFrame.'
        required = {'estimate', 'standard_error', 'ci_left_95', 'ci_right_95'}
        assert required.issubset(set(report.columns)), f'report должен содержать колонки {required}.'
        assert {'baseline', 'cuped'}.issubset(set(report.index)), 'Индекс report должен содержать baseline и cuped.'
        assert report.loc['cuped', 'standard_error'] < report.loc['baseline', 'standard_error'], 'CUPED SE должен быть меньше baseline SE.'
        print('✅ Задача 6 зачтена: итоговая таблица собрана корректно.')
    else:
        raise ValueError('Неизвестный номер задачи.')

## Задача 1. Посчитать коэффициент $\theta$

Посчитайте коэффициент

$$
\theta
=
\frac{
\operatorname{Cov}(Y, X)
}{
\operatorname{Var}(X)
}
$$

где:

- $Y$ — столбец `exp['y']`;
- $X$ — столбец `exp['x_pre']`.

**Требование:** посчитать коэффициент вручную через центрирование, без использования `np.cov`.

In [ ]:
x = exp["x_pre"]
y = exp["y"]
x_centered = x - x.mean()
y_centered = y - y.mean()

cov_y_x = (x_centered * y_centered).sum() / (len(x) - 1)
var_x = (x_centered ** 2).sum() / (len(x) - 1)
theta = cov_y_x / var_x

check_task(1, theta=theta)
print(f"theta = {theta:.4f}")

## Задача 2. Построить CUPED-метрику

Постройте столбец `y_cuped` по формуле

$$
Y_{\mathrm{CUPED}}
=
Y
-
\theta\left(X-\bar{X}\right)
$$

где:

- $Y$ — исходная метрика;
- $X$ — pre-period метрика;
- $\bar{X}$ — среднее значение $X$;
- $\theta$ — коэффициент CUPED-коррекции.

**Важно:** перед применением формулы переменную $X$ нужно центрировать.

In [ ]:
y_cuped = y - theta * x_centered
check_task(2, y_cuped=y_cuped)

## Задача 3. Сравнить standard error

Посчитайте baseline effect/SE и CUPED effect/SE. Убедитесь, что CUPED дает меньшую standard error.

In [ ]:
work = exp.assign(y_cuped=y_cuped)
control = work.loc[work["variant"] == "control"]
treatment = work.loc[work["variant"] == "treatment"]

control_y = control["y"]
treatment_y = treatment["y"]
baseline_effect = treatment_y.mean() - control_y.mean()
baseline_se = sqrt(
    treatment_y.var(ddof=1) / len(treatment_y)
    + control_y.var(ddof=1) / len(control_y)
)

control_cuped = control["y_cuped"]
treatment_cuped = treatment["y_cuped"]
cuped_effect = treatment_cuped.mean() - control_cuped.mean()
cuped_se = sqrt(
    treatment_cuped.var(ddof=1) / len(treatment_cuped)
    + control_cuped.var(ddof=1) / len(control_cuped)
)

check_task(3, y_cuped=y_cuped)
pd.DataFrame({
    "estimate": [baseline_effect, cuped_effect],
    "standard_error": [baseline_se, cuped_se],
}, index=["baseline", "cuped"]).round(4)

## Задача 4. Найти неправильный covariate

В данных есть `post_activity`. Объясните, почему его нельзя использовать как CUPED-covariate.

Ответ должен явно упоминать `post_activity`.

In [ ]:
answer = "post_activity нельзя использовать как ковариату CUPED."
explanation = (
    "Признак измерен после начала эксперимента и зависит от treatment. "
    "Поправка по нему может вычесть часть эффекта и сместить его оценку."
)
check_task(4, answer=answer, explanation=explanation)

## Задача 5. Связать корреляцию и снижение дисперсии

Посчитайте `rho = corr(x_pre, y)` и теоретическое ожидаемое снижение дисперсии `rho ** 2`.

In [ ]:
rho = exp["x_pre"].corr(exp["y"])
predicted_reduction = rho ** 2

check_task(5, rho=rho, predicted_reduction=predicted_reduction)
print(f"Корреляция: {rho:.4f}")
print(f"Ожидаемое снижение дисперсии: {predicted_reduction:.1%}")

## Задача 6. Собрать итоговый отчет

Соберите `report` с индексами `baseline` и `cuped` и колонками:

- `estimate`
- `standard_error`
- `ci_left_95`
- `ci_right_95`

Используйте нормальное приближение `estimate ± 1.96 * standard_error`.

In [ ]:
report = pd.DataFrame({
    "estimate": [baseline_effect, cuped_effect],
    "standard_error": [baseline_se, cuped_se],
}, index=["baseline", "cuped"])

margin = 1.96 * report["standard_error"]
report["ci_left_95"] = report["estimate"] - margin
report["ci_right_95"] = report["estimate"] + margin

check_task(6, report=report)
report.round(4)

## Результат

Стандартная ошибка оценки эффекта снизилась с 0,0892 до 0,0552,
то есть на 38,1%. Доверительный интервал стал уже.
Квадрат корреляции между `x_pre` и `y` равен 0,6167 и соответствует
снижению дисперсии метрики на 61,7%.